# Exercise 5: Annotation projection experiment

*Syntax and Universal Dependencies · version 0.1.0*

> Projected arcs can only be scored against gold annotation of *the same* target sentences. The Turkish and Azerbaijani TueCL treebanks annotate the same sentences, so they provide such pairs. Parse the Turkish side with the TurkicNLP Turkish pipeline (and, as an upper bound, use the Turkish gold trees), align words with fast_align or awesome-align, project the arcs to Azerbaijani, repair the result into a tree, and score it against the Azerbaijani gold. What fraction of arcs project correctly? Which dependency types project most reliably? How much of the error comes from alignment rather than from the Turkish parse?

**Setup.** `pip install "turkicnlp==0.5.0" "stanza==1.11.0" scipy matplotlib requests`. Stanza is pinned to 1.11.0 because the worked solutions use its Turkish (IMST), Kazakh (KTB) and Kyrgyz (KTMU) checkpoints; with another version, Stanza downloads that version's models and the numbers can change. Models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`); missing ones are downloaded. The Universal Dependencies files are the **UD 2.17** release files (tag `r2.17` on GitHub). They are downloaded once into `UD217_DIR` (default `~/.cache/ud217`) and checked against the SHA-256 hashes recorded in the book's treebank inventory. Parsers receive the **gold words** (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores.

**Design.** Sentences are paired by their shared `sent_id`. fast_align needs far more parallel text than 148 sentence pairs, and awesome-align needs a multilingual BERT model, so this notebook uses a small stand-in that suits two closely related Latin-script languages: both sides are converted to the Common Turkic Alphabet, word pairs are scored by character-bigram Dice similarity plus a small bonus for similar relative position, and a one-to-one alignment is found with the Hungarian algorithm (pairs below Dice 0.3 are dropped, except punctuation). Projection gives each aligned Azerbaijani word the image of the nearest aligned ancestor of its Turkish word; unaligned words and extra roots are attached to the projected root as `dep`, so every output is a tree. Scores are over all Azerbaijani words. Substitute awesome-align or fast_align where they are available.

In [ ]:
import collections, hashlib, math, os, random, statistics
from pathlib import Path
import requests

# Pinned UD 2.17 release files (the inventory in tables/tbl-ud217-inventory) and their SHA-256 hashes.
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {
    "UD_Turkish-BOUN/tr_boun-ud-train.conllu": "373fe39e3acdba3219521f54fa5ae768851fbb4bf9a31079ad1ca6590410650c",
    "UD_Turkish-BOUN/tr_boun-ud-test.conllu": "4f74fc0f9efdd1285d781f3efd1e98cbf39cf4433df9eb4dc14162e8ed36773b",
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Turkish-TueCL/tr_tuecl-ud-test.conllu": "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu": "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu": "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
    "UD_English-EWT/en_ewt-ud-train.conllu": "6fe5692f2e02198262483daba0626efe346dd19f0ce6c3ee7728b08af9b90a9c",
}
CACHE = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()

def ud_path(key):
    """Local copy of a pinned UD 2.17 file, downloaded once and checked against its SHA-256 hash."""
    path = CACHE / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos xpos feats head deprel")

def read(key_or_path):
    """Sentences as dicts: sent_id, text and words (integer-ID syntactic words only;
    multiword-token ranges and empty nodes are skipped)."""
    path = ud_path(key_or_path) if key_or_path in SHA256 else Path(key_or_path)
    sents, meta, words = [], {}, []
    for line in path.read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append({"sent_id": meta.get("sent_id"), "text": meta.get("text"), "words": words})
            meta, words = {}, []
        elif line.startswith("#"):
            k, _, v = line[1:].partition("=")
            meta.setdefault(k.strip(), v.strip())
        else:
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[4], c[5], int(c[6]), c[7]))
    return sents

def feats(w):
    return dict(f.split("=", 1) for f in w.feats.split("|") if "=" in f)

def children(words):
    out = collections.defaultdict(list)
    for w in words:
        out[w.head].append(w)
    return out

def subtree(words, i, kids=None):
    """IDs of word i and all its descendants."""
    kids = kids or children(words)
    out, stack = set(), [i]
    while stack:
        j = stack.pop(); out.add(j); stack.extend(k.id for k in kids[j])
    return out

def q(values, p):
    """Nearest-rank quantile, as in the chapter's arc-length table."""
    v = sorted(values)
    return v[math.ceil(len(v) * p) - 1] if v else None

def stanza_pipeline(lang, processors="tokenize,pos,lemma,depparse"):
    """A Stanza pipeline over gold words (tokenize_pretokenized=True). Uses models already on disk
    (STANZA_RESOURCES_DIR or ~/stanza_resources) and downloads them only if they are missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "depparse").exists()
    return stanza.Pipeline(lang, processors=processors, tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def parse_gold_words(nlp, sents, forms=None):
    """Parse each gold sentence from its gold words (optionally replaced by `forms`, one list per
    sentence, e.g. transliterated) and return predicted sentences with the original word forms."""
    forms = forms or [[w.form for w in s["words"]] for s in sents]
    doc = nlp(forms)
    out = []
    for s, ps in zip(sents, doc.sentences):
        assert len(ps.words) == len(s["words"]), s["sent_id"]
        out.append({"sent_id": s["sent_id"], "text": s["text"], "words": [
            Word(g.id, g.form, p.lemma or "_", p.upos or "_", p.xpos or "_", p.feats or "_", p.head, p.deprel)
            for g, p in zip(s["words"], ps.words)]})
    return out

def write_conllu(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(f"# sent_id = {s['sent_id']}\n# text = {s['text']}\n")
            for w in s["words"]:
                f.write("\t".join([str(w.id), w.form, w.lemma or "_", w.upos, w.xpos or "_", w.feats or "_",
                                   str(w.head), w.deprel, "_", "_"]) + "\n")
            f.write("\n")

def score(gold, pred):
    """UPOS, UAS and LAS (%) with the official CoNLL 2018 scorer, over gold-word input.
    Gold multiword tokens are not written, so both files have the same words and text."""
    import tempfile
    from stanza.utils import conll18_ud_eval as ev
    with tempfile.TemporaryDirectory() as tmp:
        g, p = Path(tmp) / "gold.conllu", Path(tmp) / "pred.conllu"
        write_conllu(gold, g); write_conllu(pred, p)
        r = ev.evaluate(ev.load_conllu_file(str(g)), ev.load_conllu_file(str(p)))
    return {k: round(100 * r[k].f1, 2) for k in ("Words", "UPOS", "UAS", "LAS")}

In [ ]:
from scipy.optimize import linear_sum_assignment
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

to_cts = {"tr": Transliterator("tur", Script("Latn"), Script("CTS")),
          "az": Transliterator("aze", Script("Latn"), Script("CTS"))}

def bigrams(s):
    s = f"#{s.lower()}#"
    return collections.Counter(s[i:i + 2] for i in range(len(s) - 1))

def similarity(a, b):
    """Dice coefficient of character bigrams of the two words in the Common Turkic Alphabet."""
    x, y = bigrams(a), bigrams(b)
    return 2 * sum((x & y).values()) / (sum(x.values()) + sum(y.values()))

def align(src, tgt, threshold=0.3, weight=0.3):
    """A small stand-in for fast_align/awesome-align: one-to-one alignment that maximises CTS
    bigram similarity plus a bonus for similar relative position. Returns {target id: source id}."""
    s = [to_cts["tr"].transliterate(w.form) for w in src]
    t = [to_cts["az"].transliterate(w.form) for w in tgt]
    score = [[similarity(a, b) + weight * (1 - abs(i / len(s) - j / len(t)))
              for j, b in enumerate(t)] for i, a in enumerate(s)]
    rows, cols = linear_sum_assignment([[-x for x in r] for r in score])
    return {tgt[j].id: src[i].id for i, j in zip(rows, cols)
            if similarity(s[i], t[j]) >= threshold or (src[i].upos == tgt[j].upos == "PUNCT")}

def project(src, tgt, a):
    """Project the source tree through alignment a. An aligned target word gets the image of the
    nearest aligned ancestor of its source word; unaligned words and extra roots are attached to
    the projected root with `dep`. The result is always a tree with one root."""
    inv = {v: k for k, v in a.items()}
    shead = {w.id: w.head for w in src}; srel = {w.id: w.deprel for w in src}
    head, rel = {}, {}
    for w in tgt:
        if w.id not in a:
            continue
        h = shead[a[w.id]]
        while h and h not in inv:
            h = shead[h]
        head[w.id] = inv.get(h, 0); rel[w.id] = srel[a[w.id]]
    roots = [i for i, h in head.items() if h == 0] or [tgt[-1].id]
    root = roots[0]
    for w in tgt:
        if w.id not in head or (head[w.id] == 0 and w.id != root):
            head[w.id], rel[w.id] = (0, "root") if w.id == root else (root, "dep")
    return [Word(w.id, w.form, "_", "_", "_", "_", head[w.id], rel[w.id]) for w in tgt]

def evaluate(source_trees, label):
    n = aligned = uas = las = same_upos = both = both_las = 0
    by_rel = collections.defaultdict(lambda: [0, 0])
    for sid, tr_words in source_trees.items():
        az_words = AZ_GOLD[sid]["words"]
        a = align(tr_words, az_words)
        proj = project(tr_words, az_words, a)
        for g, p in zip(az_words, proj):
            n += 1; aligned += g.id in a
            u = p.head == g.head; l = u and p.deprel.split(":")[0] == g.deprel.split(":")[0]
            uas += u; las += l
            if g.id in a:
                same_upos += TRW[sid][a[g.id]].upos == g.upos
                if g.head == 0 or g.head in a:
                    both += 1; both_las += l
            by_rel[g.deprel.split(":")[0]][0] += 1; by_rel[g.deprel.split(":")[0]][1] += l
    print(f"{label}: {n} Azerbaijani words, aligned {aligned / n:.1%} (same gold UPOS in {same_upos / aligned:.1%} of aligned pairs); "
          f"projected UAS {uas / n:.1%}, LAS {las / n:.1%}; LAS {both_las / both:.1%} on the {both} words whose gold head is also aligned")
    return by_rel

TR_GOLD = {s["sent_id"]: s for s in read("UD_Turkish-TueCL/tr_tuecl-ud-test.conllu")}
AZ_GOLD = {s["sent_id"]: s for s in read("UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu")}
shared = [k for k in TR_GOLD if k in AZ_GOLD]
TRW = {k: {w.id: w for w in TR_GOLD[k]["words"]} for k in shared}     # gold Turkish words (for UPOS)
print(f"{len(shared)} sentence pairs share a sent_id")

s0 = shared[0]
print("example:", [(w.form, TR_GOLD[s0]["words"][a - 1].form) for w in AZ_GOLD[s0]["words"]
                   for a in [align(TR_GOLD[s0]["words"], AZ_GOLD[s0]["words"]).get(w.id)] if a])

gold_rel = evaluate({k: TR_GOLD[k]["words"] for k in shared}, "from Turkish gold trees")
tr_pred = parse_gold_words(stanza_pipeline("tr"), [TR_GOLD[k] for k in shared])
pred_rel = evaluate({s["sent_id"]: s["words"] for s in tr_pred}, "from Stanza parses of the Turkish side")
print("labelled projection accuracy by Azerbaijani gold relation (gold source / parsed source), n >= 15:")
for r, (n, l) in sorted(gold_rel.items(), key=lambda x: -x[1][1] / x[1][0]):
    if n >= 15:
        print(f"   {r:10} n={n:3}  {l / n:.0%} / {pred_rel[r][1] / n:.0%}")

## Worked solution

Output of `experiments/ch08_exercises/run_ch08_exercises.py` (UD 2.17; Stanza 1.11.0):

```
148 sentence pairs share a sent_id
example: [('Qız', 'Kız'), ('yoldaşına', 'arkadaşına'), ('yazdı', 'yazdı'), ('.', '.')]
from Turkish gold trees: 912 Azerbaijani words, aligned 67.7% (same gold UPOS in 94.2% of aligned pairs); projected UAS 50.9%, LAS 39.4%; LAS 83.5% on the 430 words whose gold head is also aligned
from Stanza parses of the Turkish side: 912 Azerbaijani words, aligned 67.7% (same gold UPOS in 94.2% of aligned pairs); projected UAS 43.6%, LAS 30.9%; LAS 65.6% on the 430 words whose gold head is also aligned
labelled projection accuracy by Azerbaijani gold relation (gold source / parsed source), n >= 15:
   root       n=148  57% / 49%
   punct      n=182  49% / 44%
   nsubj      n=132  48% / 29%
   obj        n= 66  39% / 35%
   obl        n= 87  36% / 34%
   amod       n= 15  33% / 20%
   nmod       n= 40  28% / 18%
   conj       n= 26  27% / 19%
   aux        n= 18  22% / 0%
   cc         n= 15  20% / 20%
   advmod     n= 32  19% / 9%
   advcl      n= 25  16% / 4%
   xcomp      n= 15  7% / 7%
   compound   n= 31  3% / 0%
```

**How much projects.** From the Turkish gold trees, 50.9% of the Azerbaijani words receive the right head and 39.4% the right head and label. From Stanza's parses of the Turkish side, the figures fall to 43.6% and 30.9%.

**Alignment versus parse.** The aligner links 67.7% of the Azerbaijani words, and 94.2% of the linked pairs have the same gold UPOS, so the links it makes are mostly plausible. On the 430 words whose own link and whose gold head's link both exist, projection from gold trees is 83.5% correct (LAS): when both ends of an arc are aligned and the source tree is right, the structures largely agree. The loss to 39.4% overall is therefore mainly alignment coverage: words that are not cognates (*namə* 'letter' against Turkish *mektup*) or whose sentence is phrased differently cannot be linked by string similarity. The Turkish parse costs a further 8.5 LAS points over all words (39.4 → 30.9) and 17.9 points on the doubly aligned words (83.5 → 65.6).

**Which relations project.** `root` (57%), `punct`, `nsubj` (48%) and `obj` (39%) project best from gold trees; `advcl`, `xcomp` and `compound` project worst (3–16%). Compounds and light verbs differ in form between the languages, and Azerbaijani often uses a different construction for the same meaning (for example a finite complement clause *Fikr elirəm yağış yağır* where Turkish has a nominalised one, *Yağmur yağdığını zannediyorum*).

## Your turn — extension

1. Replace the stand-in aligner with awesome-align and compare alignment coverage and LAS.
2. Project from Uzbek TueCL to Azerbaijani. Does a less closely related source lose more through alignment or through structure?